# Credit Risk IFRS9 Project
## Notebook 01 — Data download, loading and initial inspection

This notebook downloads and prepares the Lending Club dataset that will be used as a proxy retail credit portfolio for credit risk modelling.

Main tasks:

1. Download the Lending Club loan dataset
2. Load the dataset into pandas
3. Perform an initial inspection of the data
4. Explore dataset structure and key variables
5. Prepare the dataset for subsequent analysis

## Dataset selection

This project uses the **Lending Club loan dataset** as a proxy for a retail credit portfolio. Lending Club was a peer-to-peer lending platform that issued large volumes of consumer loans, making its publicly available loan-level dataset widely used in credit risk research and modelling.

The dataset is well suited for this project for several reasons:

- It contains **loan-level observations**, allowing modelling of borrower behaviour and credit risk.
- It includes a wide range of **financial and borrower characteristics** useful for predictive modelling.
- It provides the **loan outcome variable (loan_status)**, which enables the construction of a default indicator for credit risk models.
- Its structure resembles the type of **retail lending portfolios** typically analysed in credit risk frameworks such as IFRS 9.

Although the dataset does not represent a real bank portfolio, it provides a realistic structure for building a credit risk modelling pipeline.

## Dataset content

Each row in the dataset represents an individual loan issued by Lending Club. The variables describe different aspects of the borrower, the loan contract and the credit history of the borrower.

The dataset includes several categories of variables.

**Borrower characteristics**

- annual income  
- employment length  
- home ownership status  
- debt-to-income ratio  

**Loan characteristics**

- loan amount  
- interest rate  
- loan term  
- loan purpose  
- grade assigned by the platform  

**Credit information**

- number of open credit lines  
- credit history length  
- revolving credit utilisation  
- delinquency indicators  

**Loan outcome**

- `loan_status`, which indicates whether the loan was fully paid, defaulted or charged off.

In this project, the dataset will be used to construct a simplified credit risk modelling framework including:

- Probability of Default (PD)
- Loss Given Default (LGD)
- Exposure at Default (EAD)
- Expected Credit Loss (ECL)

The first step consists of loading the dataset and performing an initial inspection of its structure and variables.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import subprocess

In [10]:
# Define project root as the current working directory
project_root = Path.cwd()

# Define raw and processed data directories relative to the repository
data_raw_path = project_root / "data" / "raw"
data_processed_path = project_root / "data" / "processed"

# Ensure the folders exist
data_raw_path.mkdir(parents=True, exist_ok=True)
data_processed_path.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data path:", data_raw_path)
print("Processed data path:", data_processed_path)

Project root: /home/apalo/Credit-Risk-Ifrs9
Raw data path: /home/apalo/Credit-Risk-Ifrs9/data/raw
Processed data path: /home/apalo/Credit-Risk-Ifrs9/data/processed


## Data access

The dataset used in this project is downloaded from Kaggle using the Kaggle API.

To reproduce this workflow, each user must configure their own Kaggle credentials by placing a valid `kaggle.json` file in:

`~/.kaggle/kaggle.json`

The credentials file is not stored in the repository for security reasons.

In [11]:
# Kaggle dataset identifier
dataset_name = "wordsforthewise/lending-club"

# Expected dataset file
dataset_file = data_raw_path / "accepted_2007_to_2018q4.csv"

# Download dataset only if it does not exist
if not dataset_file.exists():
    
    print("Downloading Lending Club dataset from Kaggle...")
    
    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            dataset_name,
            "-p",
            str(data_raw_path),
            "--unzip"
        ],
        check=True
    )
    
    print("Dataset downloaded successfully.")

else:
    
    print("Dataset already exists locally.")

Dataset already exists locally.


## Loading the dataset

The Lending Club dataset contains more than two million loans and over one hundred variables.

To inspect the structure of the data efficiently, we first load a sample of the dataset rather than the full file.

In [15]:
# Define dataset file (compressed)
dataset_file = data_raw_path / "accepted_2007_to_2018Q4.csv.gz"

# Load a sample for initial inspection
df_sample = pd.read_csv(
    dataset_file,
    compression="gzip",
    nrows=50000,
    low_memory=False
)

print("Sample shape:", df_sample.shape)

df_sample.head()

Sample shape: (50000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Columns: 151 entries, id to settlement_term
dtypes: float64(114), int64(1), object(36)
memory usage: 57.6+ MB


In [17]:
df_sample["loan_status"].value_counts(dropna=False)

loan_status
Fully Paid            34978
Charged Off            9027
Current                5610
Late (31-120 days)      246
In Grace Period         100
Late (16-30 days)        38
Default                   1
Name: count, dtype: int64

## Initial dataset overview

The sample dataset contains 50,000 observations and 151 variables. Each observation corresponds to an individual loan issued by Lending Club.

The variables include a wide range of information describing:

- borrower characteristics
- loan contract attributes
- credit history indicators
- loan performance outcomes

The large number of variables reflects the richness of the dataset, but many of them are administrative fields or highly sparse variables that may not be useful for modelling.

Therefore, in subsequent notebooks the dataset will be reduced to a smaller set of relevant variables suitable for credit risk modelling.

In [18]:
df_sample["loan_status"].value_counts(normalize=True)

loan_status
Fully Paid            0.69956
Charged Off           0.18054
Current               0.11220
Late (31-120 days)    0.00492
In Grace Period       0.00200
Late (16-30 days)     0.00076
Default               0.00002
Name: proportion, dtype: float64

## Loan outcome distribution

The variable `loan_status` indicates the final state of the loan.

The main categories observed in the sample include:

- Fully Paid
- Charged Off
- Current
- Late (31-120 days)
- Late (16-30 days)
- In Grace Period
- Default

For credit risk modelling, this variable will later be transformed into a binary **default indicator**.

A common definition used in credit risk studies is:

Default loans:
- Charged Off
- Default
- Late (31-120 days)

Non-default loans:
- Fully Paid

Loans that are still **Current** may need to be excluded depending on the modelling framework, since their final outcome is not yet observed.

In [20]:
missing_ratio = df_sample.isna().mean().sort_values(ascending=False)

missing_ratio.head(20)

member_id                                     1.00000
sec_app_num_rev_accts                         1.00000
sec_app_open_act_il                           1.00000
sec_app_inq_last_6mths                        1.00000
sec_app_open_acc                              1.00000
sec_app_mort_acc                              1.00000
sec_app_mths_since_last_major_derog           1.00000
sec_app_collections_12_mths_ex_med            1.00000
sec_app_chargeoff_within_12_mths              1.00000
sec_app_fico_range_low                        1.00000
sec_app_earliest_cr_line                      1.00000
sec_app_revol_util                            1.00000
sec_app_fico_range_high                       1.00000
revol_bal_joint                               1.00000
desc                                          0.99994
dti_joint                                     0.99530
verification_status_joint                     0.99528
annual_inc_joint                              0.99528
orig_projected_additional_ac

In [21]:
# Count variables with more than 90% missing values
high_missing = (missing_ratio > 0.9).sum()

print("Variables with >90% missing:", high_missing)

Variables with >90% missing: 38


In [22]:
# Variables with less than 50% missing values
usable_vars = (missing_ratio < 0.5).sum()

print("Variables with <50% missing:", usable_vars)

Variables with <50% missing: 94


## Interpretation of missing value analysis

The dataset contains 151 variables. The missing value analysis shows that:

- 38 variables have more than 90% missing values.
- 94 variables have less than 50% missing values.

Variables with extremely high missing ratios are unlikely to be useful for predictive modelling and will be removed during the data preparation stage.

However, the remaining variables still require further screening. Many fields in the dataset correspond to administrative information or variables that are only available after the loan has been issued. Such variables cannot be used in predictive credit risk models because they introduce data leakage.

Therefore, the next stage of the project will focus on selecting a subset of variables that are both informative and available at the time of loan origination.

## Conclusion

In this notebook we established the initial data pipeline for the project.

The main steps completed were:

- downloading the Lending Club dataset from Kaggle
- loading the data into the project environment
- inspecting the dataset structure
- analysing the loan outcome variable
- identifying variables with large proportions of missing values

The dataset contains a large number of variables, many of which are either sparsely populated or not relevant for predictive modelling.

The next step will therefore focus on:

- selecting relevant variables
- defining the default indicator
- preparing a clean modelling dataset.

These tasks will be addressed in **Notebook 02: Data cleaning and target definition**.